# Audible Insights - Step 1: Data Cleaning and Merging

In this notebook I load the two datasets, clean them and merge them into one file.

### Import libraries

In [1]:
import pandas as pd
import numpy as np

### Load the datasets

In [2]:
df1 = pd.read_csv("../data/Audible_Catlog.csv", encoding="utf-8-sig")
df2 = pd.read_csv("../data/Audible_Catlog_Advanced_Features.csv", encoding="utf-8-sig")

print("Dataset 1 shape:", df1.shape)
print("Dataset 2 shape:", df2.shape)

Dataset 1 shape: (6368, 5)
Dataset 2 shape: (4464, 8)


In [3]:
df1.head()

,Book Name,Author,Rating,Number of Reviews,Price
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,313.0,10080.0
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,4.6,3658.0,615.0
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,4.4,20174.0,10378.0
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,4.6,4614.0,888.0
4,Life's Amazing Secrets: How to Find Balance an...,Gaur Gopal Das,4.6,4302.0,1005.0


In [4]:
df2.head()

,Book Name,Author,Rating,Number of Reviews,Price,Description,Listening Time,Ranks and Genre
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9,371.0,10080,"Over the past three years, Jay Shetty has beco...",10 hours and 54 minutes,",#1 in Audible Audiobooks & Originals (See Top..."
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,4.6,3682.0,615,Brought to you by Penguin.,3 hours and 23 minutes,",#2 in Audible Audiobooks & Originals (See Top..."
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,4.4,20306.0,10378,"In this generation-defining self-help guide, a...",5 hours and 17 minutes,",#3 in Audible Audiobooks & Originals (See Top..."
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,4.6,4678.0,888,Brought to you by Penguin.,5 hours and 35 minutes,",#5 in Audible Audiobooks & Originals (See Top..."
4,Life's Amazing Secrets: How to Find Balance an...,Gaur Gopal Das,4.6,4308.0,1005,"Stop going through life, Start growing throug...",6 hours and 25 minutes,",#6 in Audible Audiobooks & Originals (See Top..."


### Basic information

In [5]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6368 entries, 0 to 6367
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Book Name          6368 non-null   object 
 1   Author             6368 non-null   object 
 2   Rating             6368 non-null   float64
 3   Number of Reviews  5737 non-null   float64
 4   Price              6365 non-null   float64
dtypes: float64(3), object(2)
memory usage: 248.9+ KB


In [6]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4464 entries, 0 to 4463
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Book Name          4464 non-null   object 
 1   Author             4464 non-null   object 
 2   Rating             4464 non-null   float64
 3   Number of Reviews  4043 non-null   float64
 4   Price              4464 non-null   int64  
 5   Description        4458 non-null   object 
 6   Listening Time     4464 non-null   object 
 7   Ranks and Genre    4464 non-null   object 
dtypes: float64(2), int64(1), object(5)
memory usage: 279.1+ KB


### Check missing values

In [7]:
print(df1.isnull().sum())
print()
print(df2.isnull().sum())

Book Name              0
Author                 0
Rating                 0
Number of Reviews    631
Price                  3
dtype: int64

Book Name              0
Author                 0
Rating                 0
Number of Reviews    421
Price                  0
Description            6
Listening Time         0
Ranks and Genre        0
dtype: int64


### Check duplicates

In [8]:
print("Duplicate rows in dataset 1:", df1.duplicated().sum())
print("Duplicate rows in dataset 2:", df2.duplicated().sum())

print("Duplicate books (same name and author) in dataset 1:", df1.duplicated(subset=["Book Name", "Author"]).sum())
print("Duplicate books (same name and author) in dataset 2:", df2.duplicated(subset=["Book Name", "Author"]).sum())

Duplicate rows in dataset 1: 929
Duplicate rows in dataset 2: 168
Duplicate books (same name and author) in dataset 1: 965
Duplicate books (same name and author) in dataset 2: 451


### Some special values

While checking the data I found that:
- Rating is **-1** when the book has no ratings
- Listening Time and Ranks and Genre are **-1** when the value is not available

In [9]:
print("Rating = -1 in dataset 1:", (df1["Rating"] == -1).sum())
print("Rating = -1 in dataset 2:", (df2["Rating"] == -1).sum())
print("Listening Time = -1:", (df2["Listening Time"] == "-1").sum())
print("Ranks and Genre = -1:", (df2["Ranks and Genre"] == "-1").sum())

Rating = -1 in dataset 1: 631
Rating = -1 in dataset 2: 421
Listening Time = -1: 2268
Ranks and Genre = -1: 2268


### Check the Description column

Many descriptions are repeated. These are errors from web scraping (website error messages), not real book descriptions.

In [10]:
df2["Description"].value_counts().head(5)

Description
Sorry, we just need to make sure you're not a robot. For best results, please make sure your browser is accepting cookies.                                                                                                                          1093
\n\nOops!\nIt's rush hour and traffic is piling up on that page. Please try again in a short while.If you were trying to place an order, it will not have been processed at this time.\n\nGo to the Amazon.in home page to continue shopping\n\n     628
\nBy completing your purchase you agree to Audible’s  Conditions of Use, Amazon's Privacy Notice and if applicable Audible's Recurring Payment Terms.\n                                                                                               84
Brought to you by Penguin.                                                                                                                                                                                                                       

## Cleaning

### Step 1: Clean book name and author (remove extra spaces)

In [11]:
df1["Book Name"] = df1["Book Name"].str.strip()
df1["Author"] = df1["Author"].str.strip()
df2["Book Name"] = df2["Book Name"].str.strip()
df2["Author"] = df2["Author"].str.strip()

# lowercase copies, only used for matching the two files
df1["name_lower"] = df1["Book Name"].str.lower()
df1["author_lower"] = df1["Author"].str.lower()
df2["name_lower"] = df2["Book Name"].str.lower()
df2["author_lower"] = df2["Author"].str.lower()

### Step 2: Clean the Description column

In [12]:
junk_words = ["not a robot",
              "traffic is piling up",
              "by completing your purchase",
              "brought to you by",
              "exclusively from audible",
              "contains the following works"]

def clean_description(text):
    if pd.isnull(text):
        return np.nan
    text = str(text).strip()
    if len(text) < 40:
        return np.nan
    for word in junk_words:
        if word in text.lower():
            return np.nan
    return text

df2["Description"] = df2["Description"].apply(clean_description)
print("Missing descriptions after cleaning:", df2["Description"].isnull().sum())

Missing descriptions after cleaning: 2043


### Step 3: Convert Listening Time to minutes

In [13]:
def convert_to_minutes(time):
    if "hour" not in time and "minute" not in time:
        return np.nan          # values like -1 or 'Not yet known'
    hours = 0
    minutes = 0
    parts = time.split(" and ")
    for part in parts:
        if "hour" in part:
            hours = int(part.split()[0])
        if "minute" in part:
            minutes = int(part.split()[0])
    return hours * 60 + minutes

df2["Listening Minutes"] = df2["Listening Time"].apply(convert_to_minutes)
df2[["Listening Time", "Listening Minutes"]].head()

,Listening Time,Listening Minutes
0,10 hours and 54 minutes,654.0
1,3 hours and 23 minutes,203.0
2,5 hours and 17 minutes,317.0
3,5 hours and 35 minutes,335.0
4,6 hours and 25 minutes,385.0


### Step 4: Get genres from 'Ranks and Genre'

Example value:
`,#1 in Audible Audiobooks & Originals (See Top 100 ...),#1 in Personal Success,#1 in Stress Management`

- The first item is the overall Audible rank
- The other items are the genres

In [14]:
def get_genres(text):
    if text == "-1":
        return ""
    genres = []
    items = text.split(",#")
    for item in items:
        if " in " in item:
            genre = item.split(" in ", 1)[1]
            genre = genre.split("(")[0].strip()
            if genre != "Audible Audiobooks & Originals" and genre not in genres:
                genres.append(genre)
    return "|".join(genres)

def get_overall_rank(text):
    if text == "-1":
        return np.nan
    items = text.split(",#")
    for item in items:
        if " in Audible Audiobooks & Originals" in item:
            rank = item.split(" in ")[0].replace(",", "").replace("#", "").replace("Free", "").strip()
            return int(rank)
    return np.nan

df2["Genres"] = df2["Ranks and Genre"].apply(get_genres)
df2["Overall Rank"] = df2["Ranks and Genre"].apply(get_overall_rank)
df2[["Genres", "Overall Rank"]].head()

,Genres,Overall Rank
0,Personal Success|Stress Management|Society & C...,1.0
1,Meditation|Self-Esteem|Personal Success,2.0
2,Personal Success|Personal Development & Self-Help,3.0
3,Psychology|Stress Management|Personal Success,5.0
4,Literary Essays|Essays|Spiritualism,6.0


### Step 5: Remove duplicates

For dataset 2, I keep the row that has the most information (genre, description and listening time).
For dataset 1, I keep the row with the most reviews.

In [15]:
df2["info_count"] = (df2["Genres"] != "").astype(int) + df2["Description"].notnull().astype(int) + df2["Listening Minutes"].notnull().astype(int)
df2 = df2.sort_values(by=["info_count", "Number of Reviews"], ascending=False)
df2 = df2.drop_duplicates(subset=["name_lower", "author_lower"], keep="first")
df2 = df2.drop(columns=["info_count"])

df1 = df1.sort_values(by="Number of Reviews", ascending=False)
df1 = df1.drop_duplicates(subset=["name_lower", "author_lower"], keep="first")

print("Dataset 1 after removing duplicates:", df1.shape)
print("Dataset 2 after removing duplicates:", df2.shape)

Dataset 1 after removing duplicates: (5402, 7)
Dataset 2 after removing duplicates: (4011, 13)


## Merge the two datasets

I use an **outer join** on book name and author so that no book is lost.

In [16]:
df2 = df2[["name_lower", "author_lower", "Book Name", "Author", "Rating", "Number of Reviews", "Price",
           "Description", "Listening Minutes", "Overall Rank", "Genres"]]

df = pd.merge(df2, df1, on=["name_lower", "author_lower"], how="outer", suffixes=("", "_df1"), indicator=True)

print(df["_merge"].value_counts())
print("Shape after merge:", df.shape)

_merge
both          3350
right_only    2052
left_only      661
Name: count, dtype: int64
Shape after merge: (6063, 17)


Rating, Number of Reviews and Price are in both files.
I take the value from dataset 2, and if it is missing I take it from dataset 1.

In [17]:
for col in ["Book Name", "Author", "Rating", "Number of Reviews", "Price"]:
    df[col] = df[col].fillna(df[col + "_df1"])

df = df.drop(columns=["Book Name_df1", "Author_df1", "Rating_df1", "Number of Reviews_df1", "Price_df1",
                      "_merge", "name_lower", "author_lower"])
df.head()

,Book Name,Author,Rating,Number of Reviews,Price,Description,Listening Minutes,Overall Rank,Genres
0,"""Don't You Know Who I Am?"": How to Stay Sane i...",Ramani S. Durvasula PhD,4.8,170.0,836.0,NaN,NaN,NaN,NaN
1,#Girlboss,Sophia Amoruso,4.5,2272.0,615.0,"Penguin presents the unabridged downloadable, ...",NaN,NaN,
2,#TheRealCinderella: #BestFriendsForever Series...,Yesenia Vargas,4.3,179.0,586.0,NaN,NaN,NaN,
3,10 Bedtime Stories For Little Kids,div.,-1.0,NaN,376.0,NaN,NaN,NaN,
4,10 Essential Pieces of Literature,Khalil Gibran,-1.0,NaN,32.0,NaN,5264.0,759.0,Classic Literature|Classic Fiction


## Handle missing values

- **Rating -1** means not rated, so I change it to NaN (I don't want to create fake ratings)
- **Number of Reviews** missing means no reviews, so I fill it with 0
- **Price** has very few missing values, so I fill them with the median
- **Genres** missing are filled with 'Unknown'

In [18]:
df["Rating"] = df["Rating"].replace(-1, np.nan)
df["Number of Reviews"] = df["Number of Reviews"].fillna(0).astype(int)
df["Price"] = df["Price"].fillna(df["Price"].median()).astype(int)
df["Genres"] = df["Genres"].fillna("")

df["Primary Genre"] = df["Genres"].apply(lambda x: x.split("|")[0] if x != "" else "Unknown")
df["Description"] = df["Description"].fillna("")

df.isnull().sum()

Book Name               0
Author                  0
Rating                581
Number of Reviews       0
Price                   0
Description             0
Listening Minutes    3967
Overall Rank         4088
Genres                  0
Primary Genre           0
dtype: int64

### Create a text column for NLP

I join book name, genres and description into one column. I will use this later for TF-IDF.

In [19]:
df["text"] = df["Book Name"] + " " + df["Genres"].str.replace("|", " ") + " " + df["Description"]
df["text"] = df["text"].str.lower()
df["text"].head()

0    "don't you know who i am?": how to stay sane i...
1    #girlboss  penguin presents the unabridged dow...
2    #therealcinderella: #bestfriendsforever series...
3                 10 bedtime stories for little kids  
4    10 essential pieces of literature classic lite...
Name: text, dtype: object

## Final check and save

In [20]:
df = df.reset_index(drop=True)

print("Final shape:", df.shape)
print("Duplicates:", df.duplicated(subset=["Book Name", "Author"]).sum())
print("Books with rating:", df["Rating"].notnull().sum())
print("Books with description:", (df["Description"] != "").sum())
print("Books with genre:", (df["Primary Genre"] != "Unknown").sum())

Final shape: (6063, 11)
Duplicates: 0
Books with rating: 5482
Books with description: 2286
Books with genre: 2059


In [21]:
df.to_csv("../data/cleaned_books.csv", index=False)
print("Saved cleaned_books.csv")

Saved cleaned_books.csv


## Summary

- Merged both datasets using book name and author (outer join)
- Removed duplicate books
- Converted listening time to minutes
- Extracted genres and overall rank from the 'Ranks and Genre' column
- Removed scraping errors from descriptions
- Rating -1 changed to NaN, missing reviews filled with 0
- Note: the dataset does not have a publication year column